# LangSmith Tracing for non-LangChain functions



In [ ]:
%pip install -qU langchain-ollama langchain-core langchain-community --quiet
%pip install langsmith --quiet
%pip install tqdm --quiet

Import Langsmith variables

In [1]:
# Load dotenv
import os
from dotenv import load_dotenv
load_dotenv()
# LANGCHAIN_API_KEY=
# LANGCHAIN_TRACING_V2=true
# LANGCHAIN_ENDPOINT=https://api.smith.langchain.com
# LANGCHAIN_PROJECT=langsmith-observe1

True

I can also set the project manually instead of reading from a static '.env'

In [ ]:
os.environ["LANGCHAIN_PROJECT"] = "langsmith-observe2"

Import libraries including Langsmith ```traceable``` decorator to be able trace non-Langchain events

In [2]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langsmith import traceable

Initializing the model

In [3]:
llm = ChatOllama(
    model="gemma4:e4b",
    temperature=0.1,
)

Create a prompt and the chain

In [4]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful AI assistant. Respond concisely."),
    ("user", "{input}")
    ])
chain = prompt | llm 

Lets create a couple of non-Langchain functions that we can trace using the @traceable decorator

In [5]:
import time, random

@traceable
def select_question() -> str:
    """Select a random question from the list"""
    question_options = [
        "What is the colour of the sky?",
        "Explain photosynthesis in 50 words or less",
        "What is the capital of Australia?"
    ]
    idx = random.randint(0,len(question_options)-1)
    return  question_options[idx]

@traceable
def wait_a_while() -> int:
    """Return a random number of seconds to wait for"""
    time_to_wait = random.randint(1,4)
    print(f"Delaying execution for {time_to_wait} seconds")
    time.sleep(time_to_wait)
    if time_to_wait > 2:
        raise ValueError(f"Execution took {time_to_wait} seconds")
    return

Execute the chain a few times to observe the traces

In [6]:
from tqdm.auto import tqdm
for _ in tqdm(range(5)):
    question = select_question()
    print(f"\n\nQUESTION: {question}")
    try:
        wait_a_while()
    except ValueError:
        pass
    response = chain.invoke({"input": question})
    

C:\Users\DELL\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.12_qbz5n2kfra8p0\LocalCache\local-packages\Python312\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
  0%|          | 0/5 [00:00<?, ?it/s]



QUESTION: Explain photosynthesis in 50 words or less
Delaying execution for 1 seconds


 20%|██        | 1/5 [00:04<00:18,  4.59s/it]



QUESTION: Explain photosynthesis in 50 words or less
Delaying execution for 4 seconds


 40%|████      | 2/5 [00:12<00:18,  6.28s/it]

...

 60%|██████    | 3/5 [00:15<00:10,  5.10s/it]



QUESTION: What is the colour of the sky?
Delaying execution for 2 seconds


 80%|████████  | 4/5 [00:18<00:04,  4.13s/it]



QUESTION: What is the capital of Australia?
Delaying execution for 3 seconds


100%|██████████| 5/5 [00:22<00:00,  4.43s/it]


All the activity generated was recording by Langsmith

![Langsmith second run](images/langsmith-non-langchain1.png)